In [ ]:
# Install required packages (run in a separate cell/terminal if needed)
# pip install scikit-learn shap tensorflow

import shap
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import copy
from sklearn.linear_model import LinearRegression
from skimage.color import gray2rgb
import os
from PIL import Image
import joblib
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet101, ResNet152
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Conv2D, MaxPooling2D, Flatten, 
    GlobalAveragePooling2D, BatchNormalization, Activation, Add
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from sklearn.svm import SVC
from scipy.fftpack import dct
from skimage.io import imread
from skimage.transform import resize
from skimage import exposure


import time


In [ ]:
# Load the models
resnet_50_model = ResNet50(weights='imagenet', include_top=False, pooling='max')
resnet_101_model = ResNet101(weights='imagenet', include_top=False, pooling='max')
resnet_152_model = ResNet152(weights='imagenet', include_top=False, pooling='max')

In [ ]:
def features_of_single_image(I):
    x = tf.expand_dims(I, axis=0)
    x = preprocess_input(x)
    
    # Extract features from each model
    features_50 = resnet_50_model.predict(x, verbose=0)[0]
    features_101 = resnet_101_model.predict(x, verbose=0)[0]
    features_152 = resnet_152_model.predict(x, verbose=0)[0]
    
    # Concatenate the features
    concatenated_features = np.concatenate([features_50, features_101, features_152])
    
    return concatenated_features

In [ ]:
# Directory and labels
image_dir = r"/home/tapti/Desktop/Saurabh/cervical/After Augmentation"
labels = ['0 Normal', '1 Abnormal']

# Create a list to collect the data
data = []
count = 0

# Populate the list with features, labels, and image paths
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            img_path = os.path.join(subdir, file)
            img = imread(img_path)
            features = features_of_single_image(img)  # Extract concatenated features
            # Add the features, label, and image path to the list
            data.append({'features': features, 'labels': label_idx, 'image_paths': img_path})
            print(count)
            count += 1

# Convert the list of dictionaries to a DataFrame
merged_df = pd.DataFrame(data)

In [ ]:
feature_length = len(merged_df.iloc[0]['features'])
print(f"The length of the concatenated feature vector is: {feature_length}")


In [ ]:
# Function for SHAP predictions
def test_with_single_input_shap(images, clf):
    num_imgs = len(images)
    result = np.zeros((num_imgs, 2))
    for i in range(num_imgs):
        image = images[i]
        res = features_of_single_image(image)
        result[i] = clf.predict_proba([res])  
    return result



In [ ]:
def explain_image(image, clf):
    masker = shap.maskers.Image("blur(28,28)", image.shape)

    # Wrap the `test_with_single_input_7200_hot_shap` function with the classifier argument
    def wrapped_test_with_single_input(images):
        return test_with_single_input_shap(images, clf)

    # Create the SHAP explainer using the wrapped function
    explainer = shap.Explainer(wrapped_test_with_single_input, masker, output_names=['Normal', 'Abnormal'])

    # Generate SHAP values for the image
    shap_values = explainer(np.array([image]), max_evals=1000, batch_size=50, outputs=shap.Explanation.argsort.flip[:3])

    # Visualize the SHAP values
    shap.image_plot(shap_values / 255)


In [ ]:


# Separate features, labels, and image paths
X = merged_df['features'].tolist()
y = merged_df['labels'].tolist()
image_paths = merged_df['image_paths'].tolist()  # Store image paths for SHAP

# Cross-validation settings
num_splits = 5
kernel = 'linear'
print("ResNet152 with 5 fold cross-validation and linear SVM")

kfold = KFold(n_splits=num_splits, shuffle=True, random_state=50)
svm = SVC(kernel=kernel, C=100, probability=True)  # Use probability=True for SHAP

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []

for fold, (train_index, test_index) in enumerate(kfold.split(X)):
    print(f"\nFold {fold + 1}:\n")
    
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]
    image_paths_test = np.array(image_paths)[test_index]  # Get image paths for SHAP

    # Train SVM model
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    # Evaluate the model performance
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    
    # Apply SHAP explanations for each test image in this fold
    count = 0
    for image_path in image_paths_test:
        print(f"Explaining image {count + 1}...")
        image = matplotlib.image.imread(image_path)  # Load the image from file using the stored image path
        explain_image(image, svm)  # Call the SHAP explainer for each test image
        count += 1

